# 05 — Eigenvalues & Eigenvectors
## Finding the Natural Directions in Data

> **Prerequisites:** Notebooks 01–04  
> **Difficulty:** ⭐⭐⭐⭐☆ Intermediate-Advanced  
> **Running example:** Finding the main direction of variation in house data (PCA preview)

---

## Before we start: A story

Imagine you have a dog toy — a rubber ball. You can squeeze it, rotate it, stretch it.

Most directions you push it, the ball distorts in complicated ways — the input direction and output direction are completely different.

But there are a **few special directions** where if you push in that direction, the ball just **moves along the same line** (maybe stretched or compressed, but not rotated).

Those special directions are **eigenvectors**. The amount of stretch is the **eigenvalue**.

---

## Why does this matter in ML?

| Application | What eigenvalues/vectors do |
|---|---|
| **PCA** | Eigenvectors of covariance matrix = principal directions of data |
| **Spectral clustering** | Eigenvectors of graph Laplacian reveal clusters |
| **PageRank** | Google's algorithm = finding the eigenvector of the web link matrix |
| **Stability analysis** | Eigenvalues tell if an optimization will converge |
| **Understanding transformations** | The "skeleton" of what a matrix does |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## 1. The Core Equation: Av = λv

### What does it mean?
A matrix A applied to a special vector **v** gives back a **scaled version** of v:
> **Av = λv**
- **v** = the eigenvector (the special direction)
- **λ** (lambda) = the eigenvalue (how much it gets scaled)
- The direction of **v** doesn't change — only its length

### Contrast with a regular vector
- Regular vector: multiply by A → completely different direction
- Eigenvector: multiply by A → **same direction**, just scaled

### Intuition with a stretch transformation
If A stretches the x-direction by 3 and the y-direction by 1:
- The vector [1, 0] (pointing right) → becomes [3, 0] → still pointing right! ✓ Eigenvector!
- The vector [0, 1] (pointing up) → stays [0, 1] → still pointing up! ✓ Eigenvector!
- The vector [1, 1] → becomes [3, 1] → now pointing in a DIFFERENT direction → not an eigenvector!

In [ ]:
# ─────────────────────────────────────────────────────────
# EIGENVECTORS: the special directions
# ─────────────────────────────────────────────────────────

print("=== Verifying Av = λv ===")
print()

# A stretch matrix: stretch x by 3, y stays the same
A = np.array([[3., 0.],
              [0., 1.]])

print(f"Matrix A:\n{A}")
print()

# Test vector 1: [1, 0] — along x-axis
v1 = np.array([1., 0.])
Av1 = A @ v1
print(f"v1 = {v1}  (pointing right)")
print(f"A @ v1 = {Av1}  → same direction! λ = 3")
print(f"3 × v1 = {3 * v1}  → matches!")
print(f"v1 is an eigenvector with eigenvalue λ = 3")
print()

# Test vector 2: [0, 1] — along y-axis
v2 = np.array([0., 1.])
Av2 = A @ v2
print(f"v2 = {v2}  (pointing up)")
print(f"A @ v2 = {Av2}  → same direction! λ = 1")
print(f"1 × v2 = {1 * v2}  → matches!")
print(f"v2 is an eigenvector with eigenvalue λ = 1")
print()

# Test vector 3: [1, 1] — diagonal — NOT an eigenvector
v3 = np.array([1., 1.])
Av3 = A @ v3
print(f"v3 = {v3}  (diagonal)")
print(f"A @ v3 = {Av3}  → DIFFERENT direction!")
print(f"v3 is NOT an eigenvector (direction changed)")

In [ ]:
# ─────────────────────────────────────────────────────────
# NUMPY: computing eigenvalues and eigenvectors
# ─────────────────────────────────────────────────────────

print("=== numpy.linalg.eig ===")
print()

A = np.array([[3., 1.],
              [0., 2.]])

print(f"Matrix A:\n{A}")
print()

# eig returns: (eigenvalues, eigenvectors)
# eigenvalues: 1D array of λ values
# eigenvectors: 2D array where EACH COLUMN is one eigenvector
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Eigenvalues  λ:  {eigenvalues}")
print(f"Eigenvectors V (columns = eigenvectors):")
print(eigenvectors.round(4))
print()
print(f"First eigenvector:  {eigenvectors[:, 0].round(4)}  (column 0)")
print(f"Second eigenvector: {eigenvectors[:, 1].round(4)}  (column 1)")
print()

# Verify Av = λv for each pair
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]      # column i = eigenvector i
    lam = eigenvalues[i]         # eigenvalue i
    Av = A @ v                   # apply the matrix
    lv = lam * v                 # scale the vector
    print(f"Pair {i+1}: λ = {lam:.4f}")
    print(f"  A @ v   = {Av.round(4)}")
    print(f"  λ × v   = {lv.round(4)}")
    print(f"  Match:    {np.allclose(Av, lv)} ✓")
    print()

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUALIZE: which vectors get rotated vs which stay on same line
# ─────────────────────────────────────────────────────────

A = np.array([[2., 1.], [1., 2.]])  # symmetric matrix (nice eigenvectors)
val, vec = np.linalg.eig(A)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_idx, ax in enumerate(axes):
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    
    if ax_idx == 0:
        ax.set_title('BEFORE: Various vectors', fontsize=11)
        # Some random vectors
        test_vecs = [np.array([1.,0.]), np.array([0.,1.]),
                     np.array([1.,0.5]), np.array([-0.5,1.])]
        for v in test_vecs:
            ax.annotate('', xy=v*1.8, xytext=(0,0),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
        # Eigenvectors
        for i in range(2):
            ev = vec[:, i]
            ax.annotate('', xy=ev*2, xytext=(0,0),
                        arrowprops=dict(arrowstyle='->', color=['red','blue'][i], lw=2.5))
            ax.text(ev[0]*2.2, ev[1]*2.2, f'e{i+1} (λ={val[i]:.0f})',
                    color=['red','blue'][i], fontsize=11, ha='center')
    else:
        ax.set_title('AFTER: A applied — eigenvectors just STRETCH!', fontsize=11)
        test_vecs = [np.array([1.,0.]), np.array([0.,1.]),
                     np.array([1.,0.5]), np.array([-0.5,1.])]
        for v in test_vecs:
            Av = A @ (v * 1.8)
            ax.annotate('', xy=Av, xytext=(0,0),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
        for i in range(2):
            ev = vec[:, i]
            Aev = A @ (ev * 2)   # apply A to the eigenvector
            ax.annotate('', xy=Aev, xytext=(0,0),
                        arrowprops=dict(arrowstyle='->', color=['red','blue'][i], lw=2.5))
            ax.text(Aev[0]+0.15, Aev[1], f'A×e{i+1}', color=['red','blue'][i], fontsize=10)

plt.suptitle('Eigenvectors (colored) keep their direction after transformation!\nOther vectors (gray) get rotated.', fontsize=11)
plt.tight_layout()
plt.show()

---
## 2. Eigendecomposition — Factoring a Matrix

### What is it?
If we have all the eigenvalue-eigenvector pairs, we can write the matrix as:
> **A = V Λ V⁻¹**

- **V** = matrix with eigenvectors as columns
- **Λ** (Lambda) = diagonal matrix with eigenvalues on diagonal
- **V⁻¹** = inverse of V

### For symmetric matrices (covariance matrices!) it's even nicer:
> **A = V Λ Vᵀ**  (because V is orthogonal → V⁻¹ = Vᵀ)

### Why does this matter?
1. **Reveals the structure** of the transformation: the eigenvalues are the "strengths", eigenvectors are the "directions"
2. **Matrix powers are easy**: A^k = V Λ^k V⁻¹ (just raise diagonal entries to power k)
3. **PCA**: finding eigenvectors of the covariance matrix gives principal components

In [ ]:
# ─────────────────────────────────────────────────────────
# EIGENDECOMPOSITION: A = V Λ V⁻¹
# ─────────────────────────────────────────────────────────

print("=== Eigendecomposition: A = V Λ V⁻¹ ===")
print()

# Create a symmetric PD matrix (like a covariance matrix)
M = np.random.randn(4, 4)
A = M.T @ M + 2 * np.eye(4)   # guaranteed symmetric, positive definite

print(f"Symmetric matrix A (4×4):")
print(A.round(3))
print()

# For symmetric: use eigh (more efficient and stable than eig)
eigenvalues, V = np.linalg.eigh(A)  # eigenvalues sorted ascending
Lambda = np.diag(eigenvalues)       # put them on a diagonal

print(f"Eigenvalues: {eigenvalues.round(4)}")
print(f"All positive: {(eigenvalues > 0).all()}  ← must be true for PD matrix")
print()
print(f"Eigenvector matrix V:")
print(V.round(3))
print()

# Reconstruct A from eigendecomposition
A_reconstructed = V @ Lambda @ V.T  # for symmetric: V⁻¹ = Vᵀ
print(f"Reconstruction error ||A - VΛVᵀ||: {np.linalg.norm(A - A_reconstructed):.2e}")
print(f"(near zero = perfect reconstruction ✓)")
print()
print(f"Eigenvectors are orthogonal (VᵀV = I): {np.allclose(V.T @ V, np.eye(4))}")

In [ ]:
# ─────────────────────────────────────────────────────────
# USEFUL PROPERTIES
# ─────────────────────────────────────────────────────────

A = np.array([[4., 2.], [1., 3.]])
val, _ = np.linalg.eig(A)

print("=== Key properties of eigenvalues ===")
print(f"Matrix A:\n{A}")
print(f"Eigenvalues: {val.round(4)}")
print()
print(f"Sum of eigenvalues = {val.sum():.4f}")
print(f"Trace of A (sum of diagonal) = {np.trace(A):.4f}")
print(f"  → They match! Sum(eigenvalues) always = trace(A)")
print()
print(f"Product of eigenvalues = {val.prod():.4f}")
print(f"Determinant of A       = {np.linalg.det(A):.4f}")
print(f"  → They match! Product(eigenvalues) always = det(A)")
print()
print("Use case: if any eigenvalue = 0, then det = 0 (singular matrix!)")

---
## 3. Application: PCA Preview

**Principal Component Analysis** finds the directions of **maximum variance** in data.

The math: these directions are the **eigenvectors of the covariance matrix**, sorted by their eigenvalues (largest eigenvalue = most variance).

This is why eigenvalues and eigenvectors are central to all of data science.

In [ ]:
# ─────────────────────────────────────────────────────────
# PCA PREVIEW: eigenvectors of the covariance matrix
# ─────────────────────────────────────────────────────────

print("=== PCA via Eigendecomposition ===")

# Generate correlated 2D data (like house size vs price)
t = np.random.randn(200)
x1 = t + 0.3*np.random.randn(200)   # size
x2 = 2*t + 0.3*np.random.randn(200) # price (correlated with size)
X = np.column_stack([x1, x2])
X -= X.mean(axis=0)  # center the data

# Step 1: Compute covariance matrix
Cov = (X.T @ X) / (len(X) - 1)  # shape (2, 2)
print(f"Covariance matrix:\n{Cov.round(3)}")
print("(Diagonal = variance of each feature, off-diagonal = covariance)")
print()

# Step 2: Find eigenvectors of covariance matrix
val, vec = np.linalg.eigh(Cov)  # sorted ascending
val = val[::-1]; vec = vec[:, ::-1]  # sort descending (most important first)

print(f"Eigenvalues (variance explained): {val.round(4)}")
print(f"Total variance: {val.sum():.4f}")
print(f"PC1 explains: {val[0]/val.sum()*100:.1f}% of variance")
print(f"PC2 explains: {val[1]/val.sum()*100:.1f}% of variance")
print()
print(f"PC1 direction (eigenvector 1): {vec[:,0].round(4)}")
print(f"PC2 direction (eigenvector 2): {vec[:,1].round(4)}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X[:,0], X[:,1], alpha=0.3, s=15)
for i, (color, pct) in enumerate(zip(['red','orange'], val/val.sum()*100)):
    scale = np.sqrt(val[i]) * 2
    direction = vec[:, i] * scale
    axes[0].annotate('', xy=direction, xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color=color, lw=3))
    axes[0].text(direction[0]*1.15, direction[1]*1.15,
                 f'PC{i+1}\n({pct:.0f}%)', color=color, fontsize=10, ha='center')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)
axes[0].set_title('Data with Principal Component Directions\n(eigenvectors of covariance)', fontsize=10)
axes[0].set_xlabel('Feature 1 (x₁)'); axes[0].set_ylabel('Feature 2 (x₂)')

# Project data onto PCs
X_pca = X @ vec
axes[1].scatter(X_pca[:,0], X_pca[:,1], alpha=0.3, s=15, color='orange')
axes[1].axhline(0, color='gray', lw=0.5); axes[1].axvline(0, color='gray', lw=0.5)
axes[1].set_xlabel(f'PC1 ({val[0]/val.sum()*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({val[1]/val.sum()*100:.1f}% variance)')
axes[1].set_title('After PCA Transform\n(data in eigenvector coordinates)', fontsize=10)
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)

plt.suptitle('PCA = Find the eigenvectors of the covariance matrix', fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary

| Concept | What it is | NumPy |
|---|---|---|
| Eigenvector | Special direction that A doesn't rotate | `np.linalg.eig(A)` |
| Eigenvalue | How much that direction gets stretched | `np.linalg.eig(A)` |
| eig vs eigh | eigh is faster and more stable for symmetric matrices | `np.linalg.eigh(A)` |
| Sum λ = trace | Sum of eigenvalues = sum of diagonal | `np.trace(A)` |
| Product λ = det | Product of eigenvalues = determinant | `np.linalg.det(A)` |
| PCA | Eigenvectors of covariance = principal directions | see notebook 11 lab |

**Next: Notebook 06 — SVD** (a generalization of eigendecomposition that works on any matrix)